# Data Vortex - Round 1: Phase 1 Exploratory Data Analysis (EDA)

## 1. Objective and Scope
This notebook conducts a comprehensive Exploratory Data Analysis (EDA) on the verified cleaned datasets:
1. `data/cleaned/Social_Engine_Users_Cleaned.csv` (1,500 user profile records)
2. `data/cleaned/Social_Engine_Posts_Cleaned.csv` (12,000 post interaction records)

### Analytical Guidelines:
- Every analysis addresses a specific descriptive or behavioral question.
- Missing values are reported transparently and never imputed.
- Descriptive statistical observations are clearly distinguished from causal claims.
- Synthetic data characteristics (e.g., uniform distributions, independent categorical attributes) are documented factually.

In [ ]:
import os
import re
from collections import Counter
import pandas as pd
import numpy as np

# Paths to cleaned datasets
USERS_PATH = os.path.join("..", "data", "cleaned", "Social_Engine_Users_Cleaned.csv")
POSTS_PATH = os.path.join("..", "data", "cleaned", "Social_Engine_Posts_Cleaned.csv")

df_users = pd.read_csv(USERS_PATH)
df_posts = pd.read_csv(POSTS_PATH)

print(f"Users Dataset: {df_users.shape[0]:,} rows x {df_users.shape[1]} columns")
print(f"Posts Dataset: {df_posts.shape[0]:,} rows x {df_posts.shape[1]} columns")

## 2. Dataset Overview & Structural Summary
Review schemas, completeness, unique identifiers, and date boundaries.

In [ ]:
overview_data = [
    {"Dataset": "Users", "Rows": len(df_users), "Cols": df_users.shape[1], "Unique IDs": df_users['user_id'].nunique(), "Missing Values": df_users.isnull().sum().sum(), "Date Range": f"{df_users['account_created'].min()} to {df_users['account_created'].max()}"},
    {"Dataset": "Posts", "Rows": len(df_posts), "Cols": df_posts.shape[1], "Unique IDs": df_posts['post_id'].nunique(), "Missing Values": df_posts.isnull().sum().sum(), "Date Range": f"{df_posts['timestamp'].min()} to {df_posts['timestamp'].max()}"}
]
pd.DataFrame(overview_data)

## 3. User Demographic & Profile Analysis

### 3.1 Geographic Distribution (`location`)
Evaluate user concentration across the 33 metropolitan regions.

In [ ]:
loc_counts = df_users['location'].value_counts()
loc_summary = pd.DataFrame({
    "User Count": loc_counts,
    "Percentage (%)": (loc_counts / len(df_users) * 100).round(2)
})
print(f"Total Unique Locations: {df_users['location'].nunique()}")
print("Top 10 Locations:")
print(loc_summary.head(10))
print("\nBottom 5 Locations:")
print(loc_summary.tail(5))

### 3.2 Language Profile (`language`)
Evaluate distribution across the 10 ISO 639-1 language codes.

In [ ]:
lang_counts = df_users['language'].value_counts()
pd.DataFrame({
    "User Count": lang_counts,
    "Percentage (%)": (lang_counts / len(df_users) * 100).round(2)
})

### 3.3 Follower Count Distribution (`follower_count`)
Assess parametric and non-parametric summary statistics to characterize the distribution shape.

In [ ]:
fc = df_users['follower_count']
fc_stats = pd.DataFrame({
    "Metric": ["Count", "Mean", "Std Dev", "Min", "10th Percentile", "25th Percentile (Q1)", "50th Percentile (Median)", "75th Percentile (Q3)", "90th Percentile", "Max", "IQR", "Skewness", "Kurtosis"],
    "Value": [
        f"{fc.count():,}", f"{fc.mean():.2f}", f"{fc.std():.2f}", f"{fc.min():,}",
        f"{fc.quantile(0.10):,.1f}", f"{fc.quantile(0.25):,.1f}", f"{fc.median():,.1f}", f"{fc.quantile(0.75):,.1f}",
        f"{fc.quantile(0.90):,.1f}", f"{fc.max():,}", f"{fc.quantile(0.75) - fc.quantile(0.25):,.1f}",
        f"{fc.skew():.4f}", f"{fc.kurt():.4f}"
    ]
})
fc_stats

### 3.4 Account Creation Dynamics (`account_created`)
Track registration seasonality over the 2023 calendar year.

In [ ]:
df_users['acc_month'] = pd.to_datetime(df_users['account_created']).dt.to_period('M')
monthly_regs = df_users['acc_month'].value_counts().sort_index()
pd.DataFrame({"Registrations": monthly_regs, "Share (%)": (monthly_regs / len(df_users) * 100).round(2)})

## 4. Post Activity Analysis

### 4.1 Activity by Platform (`platform`)
Quantify posts across social networks, with missing platforms explicitly tracked.

In [ ]:
plat_freq = df_posts['platform'].value_counts(dropna=False)
pd.DataFrame({
    "Platform": plat_freq.index.fillna("[Missing Platform]"),
    "Post Count": plat_freq.values,
    "Share of Total (%)": (plat_freq.values / len(df_posts) * 100).round(2)
})

### 4.2 Temporal Post Volume
Analyze post frequency by month, day of week, and hour.

In [ ]:
df_posts['post_dt'] = pd.to_datetime(df_posts['timestamp'])
df_posts['post_month'] = df_posts['post_dt'].dt.to_period('M')
df_posts['day_name'] = df_posts['post_dt'].dt.day_name()

print("=== POSTS PER MONTH (MAY 2024 - APR 2025) ===")
monthly_posts = df_posts['post_month'].value_counts().sort_index()
print(pd.DataFrame({"Posts": monthly_posts, "Share (%)": (monthly_posts / len(df_posts) * 100).round(2)}))

print("\n=== POSTS BY DAY OF WEEK ===")
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_posts = df_posts['day_name'].value_counts().reindex(dow_order)
print(pd.DataFrame({"Posts": dow_posts, "Share (%)": (dow_posts / len(df_posts) * 100).round(2)}))

### 4.3 User Activity Concentration (Posts per User)
Inspect the distribution of posting frequency across users.

In [ ]:
posts_per_user = df_posts['user_id'].value_counts()
ppu_stats = pd.DataFrame({
    "Metric": ["Active Users", "Mean Posts/User", "Median Posts/User", "Min Posts/User", "Max Posts/User", "Std Dev"],
    "Value": [len(posts_per_user), f"{posts_per_user.mean():.2f}", f"{posts_per_user.median():.1f}", posts_per_user.min(), posts_per_user.max(), f"{posts_per_user.std():.2f}"]
})
print(ppu_stats)
print("\nDistribution of Posts per User:")
print(posts_per_user.value_counts().sort_index())

## 5. Engagement Metrics Analysis (`likes`, `shares`, `comments`)
Detailed statistical analysis of interaction metrics.

In [ ]:
engagement_summary = []
for metric in ['likes', 'shares', 'comments']:
    s = df_posts[metric].dropna()
    engagement_summary.append({
        "Metric": metric,
        "Sample Count (n)": len(s),
        "Missing Count": df_posts[metric].isnull().sum(),
        "Missing (%)": f"{df_posts[metric].isnull().sum() / len(df_posts) * 100:.2f}%",
        "Mean": round(s.mean(), 2),
        "Median": round(s.median(), 2),
        "Std Dev": round(s.std(), 2),
        "Min": s.min(),
        "25th (Q1)": s.quantile(0.25),
        "75th (Q3)": s.quantile(0.75),
        "Max": s.max(),
        "Skewness": round(s.skew(), 4),
        "Kurtosis": round(s.kurt(), 4)
    })
pd.DataFrame(engagement_summary)

## 6. Platform Engagement Breakdown
Compare engagement performance across social networks, noting exact sample sizes.

In [ ]:
plat_summary = df_posts.groupby('platform', dropna=False).agg(
    total_posts=('post_id', 'count'),
    likes_obs=('likes', 'count'),
    likes_mean=('likes', 'mean'),
    likes_median=('likes', 'median'),
    shares_mean=('shares', 'mean'),
    shares_median=('shares', 'median'),
    comments_mean=('comments', 'mean'),
    comments_median=('comments', 'median')
).reset_index()
plat_summary['platform'] = plat_summary['platform'].fillna("[Missing Platform]")
plat_summary.round(2)

## 7. Follower Count vs. Post Engagement
Investigate linear and monotonic relationships between user followers and post interactions.

In [ ]:
merged_df = df_posts.merge(df_users[['user_id', 'follower_count', 'location', 'language']], on='user_id', how='inner')

corr_matrix = merged_df[['follower_count', 'likes', 'shares', 'comments']].corr(method='pearson')
spearman_matrix = merged_df[['follower_count', 'likes', 'shares', 'comments']].corr(method='spearman')

print("=== PEARSON CORRELATION MATRIX ===")
print(corr_matrix.round(4))
print("\n=== SPEARMAN RANK CORRELATION MATRIX ===")
print(spearman_matrix.round(4))

# Follower Quartile Tier Analysis
merged_df['follower_quartile'] = pd.qcut(merged_df['follower_count'], q=4, labels=['Q1 (Low)', 'Q2 (Mid-Low)', 'Q3 (Mid-High)', 'Q4 (High)'])
tier_engagement = merged_df.groupby('follower_quartile', observed=False).agg(
    user_count=('user_id', 'nunique'),
    post_count=('post_id', 'count'),
    likes_mean=('likes', 'mean'),
    shares_mean=('shares', 'mean'),
    comments_mean=('comments', 'mean')
).round(2)
print("\nEngagement Across Follower Quartiles:")
print(tier_engagement)

## 8. Missing Data Distribution & Pattern Analysis
Analyze missingness across months and platforms to evaluate Missing Completely At Random (MCAR) mechanisms.

In [ ]:
df_posts['missing_platform'] = df_posts['platform'].isnull()
df_posts['missing_text'] = df_posts['text_content'].isnull()
df_posts['missing_likes'] = df_posts['likes'].isnull()

print("=== MISSINGNESS BY POST MONTH (% OF MONTHLY POSTS) ===")
monthly_missing = df_posts.groupby('post_month')[['missing_platform', 'missing_text', 'missing_likes']].mean() * 100
print(monthly_missing.round(2))

print("\n=== MISSINGNESS BY PLATFORM (%) ===")
plat_missing = df_posts.groupby('platform', dropna=False)[['missing_text', 'missing_likes']].mean() * 100
print(plat_missing.round(2))

## 9. Text Content Linguistic & Semantic Analysis
Examine length distribution, top hashtags, and user mentions.

In [ ]:
valid_texts = df_posts['text_content'].dropna()
text_lengths = valid_texts.str.len()

print(f"Text Length Statistics: Min={text_lengths.min()}, Max={text_lengths.max()}, Mean={text_lengths.mean():.1f}, Median={text_lengths.median():.1f}")

# Extract hashtags and mentions
all_hashtags = []
all_mentions = []
for t in valid_texts:
    all_hashtags.extend(re.findall(r'#(\w+)', t))
    all_mentions.extend(re.findall(r'@(\w+)', t))

print(f"\nUnique Hashtags ({len(set(all_hashtags))}): Top 10:")
print(Counter(all_hashtags).most_common(10))
print(f"\nUnique Mentions ({len(set(all_mentions))}): Top 10:")
print(Counter(all_mentions).most_common(10))

## 10. Summary of Exploratory Analysis
All statistical metrics and cross-tabulations generated successfully.